# Three-phase FMU inside DPsim: MATLAB comparison

This runs the actual MATLAB-exported FMI 3.0 Model Exchange FMU through DPsim's general SSN component and MNA solver. The FMU supplies the six-state branch equations; the C++ example supplies the external 230 V RMS, 50 Hz abc voltage source. No dedicated resistor, inductor or capacitor components are constructed.

The benchmark is three independent series RLC branches in grounded wye. A separate automated fixture tests cross-phase coupling, feedthrough, shuffled variable declarations, unbalanced excitation, nonzero initial states and two ungrounded terminals. This remains a homogeneous LTI importer; arbitrary nonlinear Modelica FMUs are outside its scope.

Use the **DPsim Python** kernel and **Run All**. The notebook saves plots, metrics and the run log in `outputs/fmi-3ph-notebook`. Saving an executed `.ipynb` keeps its inline plots embedded.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path("/Volumes/SSDMatthias/development/dpsim")
MATLAB_DIR = Path("/Volumes/SSDMatthias/rlc_3ph_fmu_test")
FMU = MATLAB_DIR / "dpsim_rlc_3ph_me.fmu"
EXECUTABLE = REPO / "build-macos-check/dpsim-fmi/FMI_EMT_Ph3_LinearSSN"
OUTPUT = REPO / "outputs/fmi-3ph-notebook"
OUTPUT.mkdir(parents=True, exist_ok=True)
for path in (
    FMU,
    EXECUTABLE,
    MATLAB_DIR / "reference_trapezoidal.csv",
    MATLAB_DIR / "reference_simulink.csv",
):
    assert path.is_file(), f"Missing file: {path}"
print("FMU:", FMU)
print("Results:", OUTPUT)

## Run the imported model

Input/current channel names explicitly define the abc electrical mapping. They do not define RLC equations or prescribe the FMU state count. Phase B/C begin at nonzero voltage, so those initial voltages are supplied to FMI initialization and retained in the SSN trapezoidal history.

In [ ]:
run = subprocess.run(
    [str(EXECUTABLE), str(FMU)], cwd=OUTPUT, text=True, capture_output=True
)
(OUTPUT / "run.log").write_text(run.stdout + run.stderr)
assert run.returncode == 0, run.stdout + run.stderr
print(run.stdout)
actual = pd.read_csv(OUTPUT / "logs/fmi_rlc_3ph.csv", skipinitialspace=True)
actual.columns = actual.columns.str.strip()
trap = pd.read_csv(MATLAB_DIR / "reference_trapezoidal.csv")
simulink = pd.read_csv(MATLAB_DIR / "reference_simulink.csv")
display(actual.head())

## Check all three phases

The trapezoidal reference uses the same 10 µs integration rule as DPsim. Agreement is checked within the logger's six-decimal rounding. Simulink uses tightly toleranced adaptive `ode45`; its traces are interpolated to DPsim timestamps, so that comparison includes interpolation and integration error. The independent MNA check verifies each source current cancels the corresponding imported branch current.

In [ ]:
spec = importlib.util.spec_from_file_location(
    "compare_3ph", REPO / "dpsim-fmi/tests/compare_reference_3ph.py"
)
comparison = importlib.util.module_from_spec(spec)
spec.loader.exec_module(comparison)
report = comparison.validate(OUTPUT / "logs/fmi_rlc_3ph.csv", MATLAB_DIR)
(OUTPUT / "comparison.json").write_text(json.dumps(report, indent=2))
metrics = pd.concat(
    {key: pd.DataFrame(report[key]).T for key in ("trapezoidal", "simulink")},
    names=["reference", "signal"],
)
metrics.to_csv(OUTPUT / "metrics.csv")
display(metrics)
print("Maximum per-phase KCL residual [A]:", report["max_kcl_residual_A"])
print("Maximum current sum [A], including CSV rounding:", report["max_current_sum_A"])

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True)
for column, phase in enumerate("abc"):
    for row, (quantity, unit) in enumerate((("u", "V"), ("i", "A"), ("v_C", "V"))):
        key = f"{quantity}_{phase}"
        ax = axes[row, column]
        ax.plot(
            simulink.time_s * 1000,
            simulink[f"{key}_{unit}"],
            color="0.5",
            label="MATLAB ode45",
            linewidth=2.2,
        )
        ax.plot(
            trap.time_s * 1000,
            trap[f"{key}_{unit}"],
            "--",
            label="MATLAB trapezoidal",
            linewidth=1.4,
        )
        ax.plot(actual.time * 1000, actual[key], label="DPsim FMU / SSN", linewidth=0.9)
        ax.set(title=f"{quantity}, phase {phase.upper()}", ylabel=unit)
        ax.grid(alpha=0.25)
        if row == 2:
            ax.set_xlabel("Time [ms]")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Three-phase RLC: imported FMU solved inside DPsim MNA", fontsize=15)
fig.tight_layout()
fig.savefig(OUTPUT / "waveforms.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for row, (quantity, unit) in enumerate((("i", "A"), ("v_C", "V"))):
    for column, (reference, name) in enumerate(
        ((trap, "MATLAB trapezoidal"), (simulink, "MATLAB ode45, interpolated"))
    ):
        ax = axes[row, column]
        for phase in "abc":
            key = f"{quantity}_{phase}"
            interpolated = np.interp(
                actual.time, reference.time_s, reference[f"{key}_{unit}"]
            )
            ax.plot(
                actual.time * 1000,
                actual[key] - interpolated,
                label=f"Phase {phase.upper()}",
                linewidth=0.8,
            )
        ax.set(title=f"DPsim minus {name}", ylabel=f"{quantity} error [{unit}]")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
        if row == 1:
            ax.set_xlabel("Time [ms]")
fig.tight_layout()
fig.savefig(OUTPUT / "errors.png", dpi=180, bbox_inches="tight")
plt.show()

The waveform agreement validates this six-state MATLAB benchmark and its explicit abc mapping. It does not prove that every exported FMU is homogeneous LTI or that every exporter uses compatible initialization. The self-contained C++ tests additionally validate coupled matrices and MNA terminal signs.

To share plots without rerunning, send a saved, executed notebook or its HTML export. The comparison reads the FMU and reference CSV files for execution; rendered notebook outputs remain embedded when opened elsewhere.